# 23. PM10 다변량 LSTM: PM2.5 함께 사용하기

이번 장에서는 PM10 하나만 쓰지 않고 PM2.5도 함께 입력으로 사용합니다.

핵심 변화:

```text
단변량 입력: (samples, 24, 1)
다변량 입력: (samples, 24, 2)
```

정답은 여전히 다음 시간 PM10 하나입니다.

## 1. 라이브러리 불러오기

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

from keras.models import Sequential
from keras.layers import Input, LSTM, Dense, Dropout
from keras.callbacks import EarlyStopping

## 2. 데이터 읽기와 정리

In [ ]:
DATA_PATH = Path(r"C:\work\dataset\seoul_pm10.csv")
TARGET_AREA = "강남구"

df = pd.read_csv(DATA_PATH, encoding="cp949")
df["date"] = pd.to_datetime(df["date"], errors="coerce")

area_df = df[df["area"] == TARGET_AREA].copy()
area_df = area_df.sort_values("date")

# PM10과 PM2.5 결측치를 앞 시간 값으로 채웁니다.
area_df["pm10_filled"] = area_df["pm10"].ffill()
area_df["pm25_filled"] = area_df["pm2.5"].ffill()

# 두 값 중 하나라도 남은 결측치가 있으면 제거합니다.
area_df = area_df.dropna(subset=["pm10_filled", "pm25_filled"])

print("선택 지역:", TARGET_AREA)
print("정리 후 shape:", area_df.shape)
area_df[["date", "pm10_filled", "pm25_filled"]].head()

## 3. PM10과 PM2.5 함께 보기

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(area_df["date"], area_df["pm10_filled"], label="PM10", alpha=0.8)
plt.plot(area_df["date"], area_df["pm25_filled"], label="PM2.5", alpha=0.8)
plt.title("PM10 and PM2.5")
plt.xlabel("Date")
plt.ylabel("Value")
plt.legend()
plt.tight_layout()
plt.show()

## 4. 입력 특성 배열 만들기

In [ ]:
# 이번 장의 입력 특성은 PM10과 PM2.5 두 개입니다.
feature_columns = ["pm10_filled", "pm25_filled"]

# target은 다음 시간 PM10입니다.
target_column = "pm10_filled"

values = area_df[feature_columns].values

print("values shape:", values.shape)
print("앞 5행:")
print(values[:5])

## 5. 시간 순서 train/validation split

In [ ]:
split_index = int(len(values) * 0.8)

train_values = values[:split_index]
val_values = values[split_index:]

print("train_values shape:", train_values.shape)
print("val_values shape:", val_values.shape)

## 6. 다변량 MinMaxScaler 적용

In [ ]:
# MinMaxScaler는 각 컬럼별로 0~1 범위 변환 기준을 학습합니다.
scaler = MinMaxScaler()

# train 데이터에만 fit합니다.
train_scaled = scaler.fit_transform(train_values)

# validation 데이터에는 transform만 적용합니다.
val_scaled = scaler.transform(val_values)

print("train_scaled shape:", train_scaled.shape)
print("val_scaled shape:", val_scaled.shape)
print("스케일링 후 앞 5행:")
print(train_scaled[:5])

## 7. 다변량 window 데이터셋 만들기

입력 X는 과거 24시간의 PM10과 PM2.5입니다.

정답 y는 다음 시간의 PM10입니다.

In [ ]:
def make_multivariate_window_dataset(values, window_size, target_index):
    """다변량 시계열 values에서 window 입력과 다음 시점 target을 만듭니다."""
    
    X = []
    y = []
    
    for i in range(len(values) - window_size):
        # 과거 window_size시간의 모든 특성을 입력으로 사용합니다.
        X.append(values[i:i + window_size, :])
        
        # 다음 시간의 target_index 컬럼만 정답으로 사용합니다.
        y.append(values[i + window_size, target_index])
    
    return np.array(X), np.array(y)


WINDOW_SIZE = 24

# feature_columns에서 pm10_filled의 위치입니다.
target_index = feature_columns.index(target_column)

X_train, y_train = make_multivariate_window_dataset(train_scaled, WINDOW_SIZE, target_index)
X_val, y_val = make_multivariate_window_dataset(val_scaled, WINDOW_SIZE, target_index)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_val shape:", X_val.shape)
print("y_val shape:", y_val.shape)

## 8. shape 읽기

In [ ]:
samples, timesteps, features = X_train.shape

print("samples:", samples)
print("timesteps:", timesteps)
print("features:", features)
print("특성 이름:", feature_columns)

## 9. 첫 번째 window 확인

In [ ]:
first_window = X_train[0]

print("첫 번째 window shape:", first_window.shape)
print("첫 번째 window 앞 5시간:")
print(first_window[:5])

print("\n첫 번째 정답 y:", y_train[0])

## 10. LSTM 모델 만들기

In [ ]:
model = Sequential([
    # 입력 모양은 (과거 시간 길이, 특성 수)입니다.
    # 이번 장에서는 (24, 2)입니다.
    Input(shape=(WINDOW_SIZE, len(feature_columns))),
    
    # LSTM은 시간 순서가 있는 다변량 입력을 읽습니다.
    LSTM(32),
    
    Dropout(0.2),
    
    # 다음 시간 PM10 하나를 예측합니다.
    Dense(1)
])

model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

model.summary()

## 11. 모델 학습

In [ ]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)

## 12. 예측과 원래 PM10 단위 복원

In [ ]:
y_val_pred_scaled = model.predict(X_val, verbose=0)

# 다변량 scaler는 PM10, PM2.5 두 컬럼 기준으로 학습되었습니다.
# PM10 하나만 inverse_transform하려면 임시로 2컬럼 배열을 만들어 PM10 위치에 값을 넣습니다.
def inverse_pm10_only(pm10_scaled_values, scaler, target_index, feature_count):
    """스케일링된 PM10 한 컬럼을 원래 PM10 단위로 되돌립니다."""
    
    temp = np.zeros((len(pm10_scaled_values), feature_count))
    temp[:, target_index] = pm10_scaled_values.ravel()
    
    inversed = scaler.inverse_transform(temp)
    
    return inversed[:, target_index]


y_val_original = inverse_pm10_only(y_val, scaler, target_index, len(feature_columns))
y_val_pred_original = inverse_pm10_only(y_val_pred_scaled, scaler, target_index, len(feature_columns))

print("실제 PM10 앞 5개:", y_val_original[:5])
print("예측 PM10 앞 5개:", y_val_pred_original[:5])

## 13. MAE/RMSE 계산

In [ ]:
mae = mean_absolute_error(y_val_original, y_val_pred_original)
rmse = np.sqrt(mean_squared_error(y_val_original, y_val_pred_original))

print(f"다변량 LSTM MAE: {mae:.2f}")
print(f"다변량 LSTM RMSE: {rmse:.2f}")

## 14. 실제값과 예측값 비교

In [ ]:
PLOT_COUNT = 200

plt.figure(figsize=(12, 4))
plt.plot(y_val_original[:PLOT_COUNT], label="actual PM10")
plt.plot(y_val_pred_original[:PLOT_COUNT], label="predicted PM10")
plt.title("Multivariate LSTM: Actual vs Predicted PM10")
plt.xlabel("Validation time step")
plt.ylabel("PM10")
plt.legend()
plt.tight_layout()
plt.show()

## 15. 해석 문장 만들기

In [ ]:
print("해석 템플릿")
print("- 단변량 LSTM은 PM10 하나만 입력으로 사용했다.")
print("- 다변량 LSTM은 과거 24시간의 PM10과 PM2.5를 함께 입력으로 사용했다.")
print("- 입력 shape는 (samples, 24, 2)이며, 정답은 다음 시간 PM10 하나다.")
print("- PM2.5 추가가 실제로 도움이 되었는지는 단변량 모델과 MAE/RMSE로 비교해야 한다.")

## 16. 이번 장 정리

이번 장에서 배운 핵심은 다음과 같습니다.

```text
1. 다변량 시계열은 한 시점에 여러 특성을 가진다.
2. PM10과 PM2.5를 함께 쓰면 features가 2가 된다.
3. 입력 shape는 (samples, timesteps, features)다.
4. 입력 특성이 2개여도 정답이 PM10 하나라면 출력층은 Dense(1)이다.
5. 다변량 모델이 더 좋은지는 반드시 비교 실험으로 확인해야 한다.
```

## 과제

1. `X_train.shape`를 자기 말로 읽어보세요.
2. PM10과 PM2.5를 함께 쓰는 것이 왜 도움이 될 수 있는지 설명해보세요.
3. 다변량 scaler에서 PM10만 원래 단위로 되돌릴 때 임시 배열을 만든 이유를 설명해보세요.
4. 다음에 추가하고 싶은 외부 특성 3개를 적어보세요.